In [304]:
from mwprop.nemod.NE2025 import ne2025
from astropy.coordinates import SkyCoord
from astropy import units as u


In [305]:

# --- Manually input coordinates here ---
ra_str = "17:44:23.5"    # RA in hh:mm:ss
dec_str = " -47:07:59"    # Dec in dd:mm:ss

# Clean up whitespace, just in case
ra_str = ra_str.replace(" ", "")
dec_str = dec_str.replace(" ", "")

c = SkyCoord(ra_str + " " + dec_str, unit=(u.hourangle, u.deg))

_, modouts, _, _ = ne2025(
    ldeg=c.galactic.l.value,
    bdeg=c.galactic.b.value,
    dmd=50,
    ndir=-1,
    classic=False
)

print(f"RA: {ra_str}, Dec: {dec_str}")
print(f"l = {c.galactic.l.value:.4f}, b = {c.galactic.b.value:.4f}")
print("DM_MW:", round(modouts['DM'], 2))
print("Full model output:", modouts)

RA: 17:44:23.5, Dec: -47:07:59
l = 344.2205, b = -9.1836
DM_MW: 205.33
Full model output: {'l': 344.22054937854244, 'b': -9.18363499475197, 'DM/D': 50, 'ndir': -1, 'limdist': ' ', 'DIST': 50, 'DM': 205.3315416093955, 'DMz': 32.77075637330216}


For a file

In [306]:
import pandas as pd
import numpy as np
# --- Configuration ---
CSV_FILE = '/home/thsu/FRB_population/DM_distribution_real_data/dsa-110.csv'   # read from and written back to this same file

RA_COL = 'RA'      # RA column name
DEC_COL = 'Dec'    # Dec column name
DM_COL = 'DM'      # observed DM column name

DEGREES = False    # True if RA/Dec are decimal degrees, False for hh:mm:ss / dd:mm:ss
DMD = 50           # dmd passed to ne2025

def dm_mw_ne2025(ra, dec, degrees=False, dmd=50):
    '''Return NE2025 DM_MW for one position (np.nan if it can't be computed).'''
    try:
        if degrees:
            c = SkyCoord(float(ra) * u.deg, float(dec) * u.deg)
        else:
            ra_str = str(ra).replace(' ', '')
            dec_str = str(dec).replace(' ', '')
            c = SkyCoord(f'{ra_str} {dec_str}', unit=(u.hourangle, u.deg))

        _, modouts, _, _ = ne2025(
            ldeg=c.galactic.l.value,
            bdeg=c.galactic.b.value,
            dmd=dmd,
            ndir=-1,
            classic=False,
        )
        return float(modouts['DM'])
    except Exception as e:
        print(f'Could not compute DM for RA={ra}, Dec={dec}: {e}')
        return np.nan

In [307]:
df = pd.read_csv(CSV_FILE)

# Keep any pre-existing DM_ne2025 values
if 'DM_ne2025' not in df.columns:
    df['DM_ne2025'] = np.nan
df['DM_ne2025'] = pd.to_numeric(df['DM_ne2025'], errors='coerce')

def _has_value(x):
    return pd.notna(x) and str(x).strip().lower() not in ('', 'nan', 'none', '-', '--')

def _safe_dm(r, d):
    """Run the model on one position; return NaN if the coordinates can't be parsed."""
    try:
        if DEGREES:                      # numeric degrees expected
            r, d = float(r), float(d)
        return round(dm_mw_ne2025(r, d, degrees=DEGREES, dmd=DMD), 2)
    except Exception as e:
        print(f"  Skipping RA={r!r}, Dec={d!r}: {e}")
        return np.nan

if RA_COL in df.columns and DEC_COL in df.columns:
    has_coords = df[RA_COL].map(_has_value) & df[DEC_COL].map(_has_value)
    todo = df['DM_ne2025'].isna() & has_coords

    df.loc[todo, 'DM_ne2025'] = [
        _safe_dm(r, d) for r, d in zip(df.loc[todo, RA_COL], df.loc[todo, DEC_COL])
    ]
    print(f"Computed DM_ne2025 for {df.loc[todo, 'DM_ne2025'].notna().sum()} of {todo.sum()} candidate rows.")
else:
    print(f"Warning: '{RA_COL}'/'{DEC_COL}' not found; using existing DM_ne2025 only. "
          f"Available columns: {list(df.columns)}")

if DM_COL in df.columns:
    df['DM_EG'] = (pd.to_numeric(df[DM_COL], errors='coerce') - df['DM_ne2025']).round(2)
else:
    print(f"Warning: DM column '{DM_COL}' not found, so DM_EG was not added. "
          f"Available columns: {list(df.columns)}")

missing = df['DM_ne2025'].isna()
if missing.any():
    print(f"{missing.sum()} rows still have no DM_ne2025 (no existing value and no usable RA/Dec).")

df.head()
df.to_csv(CSV_FILE, index=False)

Computed DM_ne2025 for 0 of 0 candidate rows.
